In [2]:
from langchain_huggingface import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)   



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from langchain_chroma import Chroma

vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="db/chroma_db",  
)

In [38]:
query="Where is VJTI located?"

In [39]:
results=vector_store.similarity_search(query,k=2)

In [40]:
for res in results:
    print(res.page_content)

VJTI is an academically and administratively autonomous institute, but it is affiliated to the University of Mumbai. The institute is financially supported by the Government of Maharashtra.
After being awarded academic and administrative autonomy in 2004, VJTI became operational under the administration of a board of governors.[2] VJTI is also the Central Technical Institute of Maharashtra State. The institute trains students in engineering and technology at the certificate,[3] diploma, degree, post-graduate and doctoral levels.


In [41]:
context_for_llm=context = "\n\n".join([doc.page_content for doc in results])


In [42]:
prompt=f"""
Answer the question based on the given context.
context: {context_for_llm}
question: {query}
"""

In [43]:
import os
from dotenv import load_dotenv
load_dotenv() 
from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {"role": "user", "content": prompt}
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

The context does not provide information about the location of VJTI. It only mentions that VJTI is the Central Technical Institute of Maharashtra State, which suggests that it is located in Maharashtra, but the exact location is not specified.
